In [1]:
# Import pandas for working with our time-series data.
import pandas as pd

# Load the weekly demand dataset.
df = pd.read_parquet("../artifacts/weekly.parquet")

# Sort the data so every product is in chronological order.
df = df.sort_values(["item", "date"]).reset_index(drop=True)

# Find the final date in our dataset.
max_date = df["date"].max()

# Reserve the final 26 weeks as our untouched test period.
test_start = max_date - pd.Timedelta(weeks=26)

# Everything before the test period is our training/validation data.
train_df = df[df["date"] < test_start].copy()

# The final 26 weeks are our final evaluation data.
test_df = df[df["date"] >= test_start].copy()

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (4660, 3)
Test: (540, 3)


In [5]:
# Create a copy of the full dataset.
baseline_df = df.copy()

# For each product, shift demand by 52 weeks.
# This gives us the demand from approximately the same week one year earlier.
baseline_df["seasonal_naive"] = (
    baseline_df
    .groupby("item")["demand"]
    .shift(52)
)

# Keep only the final 26 weeks for evaluation.
seasonal_naive = baseline_df[
    baseline_df["date"] >= test_start
].copy()

# Display the predictions.
seasonal_naive.head()

,item,date,demand,seasonal_naive
233,1,2017-07-02,2119,2185.0
234,1,2017-07-09,2227,2298.0
235,1,2017-07-16,2261,2112.0
236,1,2017-07-23,2239,2104.0
237,1,2017-07-30,2247,2194.0


In [6]:
# Count how many test rows have a valid baseline prediction.
print(
    "Valid predictions:",
    seasonal_naive["seasonal_naive"].notna().sum()
)

# Count predictions that are missing.
print(
    "Missing predictions:",
    seasonal_naive["seasonal_naive"].isna().sum()
)

Valid predictions: 540
Missing predictions: 0


In [7]:
# Calculate the total absolute forecasting error.
absolute_error = (
    seasonal_naive["demand"]
    - seasonal_naive["seasonal_naive"]
).abs().sum()

# Calculate total actual demand.
total_actual = seasonal_naive["demand"].sum()

# Calculate WAPE.
wape = absolute_error / total_actual

print(f"Seasonal Naive WAPE: {wape:.2%}")

Seasonal Naive WAPE: 4.04%


In [8]:
# Make a copy of the dataset so we can create our baseline prediction.
baseline_df = df.copy()

# Calculate the average demand from the previous 4 weeks for each product.
# shift(1) makes sure we do NOT use the current week's demand.
baseline_df["moving_average"] = (
    baseline_df
    .groupby("item")["demand"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

# Keep only the final test period for evaluation.
moving_average = baseline_df[
    baseline_df["date"] >= test_start
].copy()

# Display the actual demand and our prediction.
moving_average[["date", "item", "demand", "moving_average"]].head(10)

,date,item,demand,moving_average
233,2017-07-02,1,2119,2071.00
234,2017-07-09,1,2227,2101.50
235,2017-07-16,1,2261,2130.25
236,2017-07-23,1,2239,2171.25
237,2017-07-30,1,2247,2211.50
238,2017-08-06,1,1947,2243.50
239,2017-08-13,1,1961,2173.50
240,2017-08-20,1,1926,2098.50
241,2017-08-27,1,1948,2020.25
242,2017-09-03,1,1887,1945.50


In [9]:
# Calculate the total absolute forecasting error.
absolute_error = (
    moving_average["demand"] - moving_average["moving_average"]
).abs().sum()

# Calculate the total actual demand.
total_actual = moving_average["demand"].sum()

# Calculate WAPE.
wape = absolute_error / total_actual

# Display the result as a percentage.
print(f"4-Week Moving Average WAPE: {wape:.2%}")

4-Week Moving Average WAPE: 6.70%
